# Does the adapter generalize around its fitting positions?

Select a **GPU runtime → Run all**. This diagnostic uses the **clean no-anchor v0** adapter (after near-band fitting), its actual fitting pairs and the same saved goal.

For up to 50 representative fitting pairs A→B, it renders 11 positions along the straight segment and nearby parallel segments at ±0.5, ±1 and ±2 simulation units. Every retained segment physically approaches the goal and stays on the same unobstructed side of the wall.

It compares **raw MSE and adapted MSE on identical fresh images**. No adapter training, world-model loading or planning episodes occur. Existing source results are read-only; new results go to a timestamped `local_interpolation_v1` folder.

Full heatmaps extend from x=40 to just inside the physical right border (about x=60). A dotted line marks the narrower reset-safe range.


In [ ]:
from google.colab import drive
from pathlib import Path
from datetime import datetime, timezone
import hashlib, json, shutil, subprocess, sys, time
import torch

drive.mount('/content/drive')
assert torch.cuda.is_available(), 'Choose a GPU runtime for the fresh encoder checks.'
DRIVE = Path('/content/drive/MyDrive')
CURRICULUM = DRIVE / 'datasets/curriculum_cost_adapter/multiband_no_anchor_v1'
PARENT = DRIVE / 'datasets/curriculum_cost_adapter/pilot_v1'
ADJACENT = DRIVE / 'datasets/curriculum_cost_adapter/adjacent_band_v1'
CLEAN = DRIVE / 'datasets/curriculum_cost_adapter/clean_pair_v1'
ENCODER_DRIVE = DRIVE / 'checkpoints/IN1K-vit.h.14-300e.encoder-only.pth.tar'
OUTPUT = DRIVE / 'datasets/curriculum_cost_adapter/local_interpolation_v1' / datetime.now(timezone.utc).strftime('%Y%m%dT%H%M%S%fZ')
REPO = Path('/content/i-jepa-wm-local-interpolation')
REPO_URL = 'https://github.com/kmylonas/i-jepa-wm.git'
SOURCE_COMMIT = '479ec97e2a4c8876e9619d8c0709619e759a7f4a'
DINO_ROOT = Path('/content/dino_wm')
DINO_COMMIT = '0a9492fa12044b852ae9e001cc74604b79c8bb0c'
ENCODER = Path('/content/curriculum-local-interpolation-artifacts/encoder.pth.tar')

def sha256(path):
    result = hashlib.sha256()
    with Path(path).open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            result.update(block)
    return result.hexdigest()

def shell(command):
    command = [str(arg) for arg in command]
    print('$', ' '.join(command), flush=True)
    with subprocess.Popen(command, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
        try:
            for line in process.stdout:
                print(line, end='', flush=True)
            code = process.wait()
        except BaseException:
            process.terminate()
            try: process.wait(timeout=10)
            except subprocess.TimeoutExpired: process.kill(); process.wait()
            raise
    if code: raise subprocess.CalledProcessError(code, command)

assert all(p.is_dir() for p in (CURRICULUM,CLEAN,PARENT,ADJACENT)), 'The no-anchor curriculum and its three sources must be in Drive.'
assert ENCODER_DRIVE.is_file(), f'Missing encoder checkpoint: {ENCODER_DRIVE}'
print('GPU:', torch.cuda.get_device_name())
print('Existing experiment:', CURRICULUM)
print('New diagnostic output:', OUTPUT)


## Load the frozen image encoder and pinned renderer

The world-model checkpoint is not needed. The checkpoint and replay used for fitting are validated before probing. The cached goal remains unchanged; this diagnostic measures spatial generalization for that one goal.


In [ ]:
if not REPO.exists():
    shell(['git', 'clone', '--branch', 'curriculum-cost-adapter', '--single-branch', REPO_URL, REPO])
else:
    assert subprocess.check_output(['git', '-C', str(REPO), 'remote', 'get-url', 'origin'], text=True).strip() == REPO_URL
    assert not subprocess.check_output(['git', '-C', str(REPO), 'status', '--porcelain'], text=True).strip(), 'Diagnostic clone has local changes.'
    shell(['git', '-C', REPO, 'fetch', 'origin', 'curriculum-cost-adapter'])
assert SOURCE_COMMIT, 'Use the published notebook with a pinned diagnostic revision.'
shell(['git', '-C', REPO, 'checkout', '--detach', SOURCE_COMMIT])
shell([sys.executable, '-m', 'pip', 'install', '-q', 'gym==0.26.2', 'omegaconf', 'pyyaml', 'scipy', 'einops', 'matplotlib', 'pillow'])
if not DINO_ROOT.exists():
    shell(['git', 'clone', 'https://github.com/gaoyuezhou/dino_wm.git', DINO_ROOT])
assert not subprocess.check_output(['git', '-C', str(DINO_ROOT), 'status', '--porcelain', '--untracked-files=no'], text=True).strip(), 'DINO source has tracked changes.'
shell(['git', '-C', DINO_ROOT, 'checkout', DINO_COMMIT])
expected_encoder = json.loads((CURRICULUM / 'experiment.json').read_text())['sources']['models']['encoder']
ENCODER.parent.mkdir(parents=True, exist_ok=True)
if not ENCODER.exists() or sha256(ENCODER) != expected_encoder:
    shutil.copy2(ENCODER_DRIVE, ENCODER)
assert sha256(ENCODER) == expected_encoder, 'Encoder differs from the experiment checkpoint.'
print('Ready: only the frozen image encoder will be loaded; no world-model checkpoint is needed.')


## Render and score the probes

Coordinates—not latents—are interpolated. Pairs are reconstructed from the exact fitting sampler and selected across episodes/separations without using their adapted costs. Incorrect endpoint rankings remain in the report.

Positions already used by fitting, positions merely present elsewhere in replay, and identical fresh renders at fitted positions are tracked separately. Sideways probes that leave the environment or cease to approach the goal are excluded and counted. Fresh encoding runs in batches of eight.


In [ ]:
started = time.monotonic()
command = [sys.executable, '-m', 'src.mylonas_ijepa_wm.curriculum_local_audit',
    '--parent-dir', PARENT, '--clean-dir', CLEAN, '--adjacent-dir', ADJACENT,
    '--curriculum-dir', CURRICULUM, '--output-dir', OUTPUT,
    '--encoder-checkpoint', ENCODER, '--dino-wm-root', DINO_ROOT,
    '--pair-count', '50', '--device', 'cuda']
# Run from the pinned clone. The audit reads existing files and writes only to OUTPUT.
command = [str(arg) for arg in command]
with subprocess.Popen(command, cwd=REPO, stdout=subprocess.PIPE, stderr=subprocess.STDOUT, text=True, bufsize=1) as process:
    try:
        for line in process.stdout:
            print(line, end='', flush=True)
        result = process.wait()
    except BaseException:
        process.terminate()
        try: process.wait(timeout=10)
        except subprocess.TimeoutExpired: process.kill(); process.wait()
        raise
if result: raise subprocess.CalledProcessError(result, command)
print(f'Local interpolation diagnostic took {(time.monotonic()-started)/60:.1f} minutes.')


## Read the summary

- **Endpoint OK:** A has higher cost than B.
- **Monotone:** no uphill steps and at least one meaningful cost decrease.
- **Uphill:** a local cost increase while moving toward B and the goal; lower is better.
- **Flat:** no meaningful decreases or increases; smooth, but not an informative direction.
- **Unseen uphill / steps:** error count restricted to comparisons between two previously unfitted observations; zero denominator means no such evidence.

Local-step tolerance is `max(1e-7, 0.001 × that method/profile cost range)`. Endpoint-order tolerance uses the absolute endpoint contrast instead, so an interior ridge cannot create a false replay/fresh mismatch. Reported fractions are descriptive, not independent statistical trials. Pairs from one trajectory and neighboring probes are related.


In [ ]:
from IPython.display import display, Image
report = json.loads((OUTPUT / 'report.json').read_text())
probes = json.loads((OUTPUT / 'probes.json').read_text())
print('Checkpoint: clean no-anchor v0', report['checkpoint_sha256'])
print('Actual pair draws during fitting:', report['sampling']['total_pair_draws'])
print('Unique sampled pairs:', report['sampling']['unique_sampled_pairs'])
print('Geometrically eligible pairs:', report['selection']['geometrically_eligible'])
print('Selected pairs:', report['selected_pairs'])
print('Offset profiles excluded by bounds/distance:', report['rejected_offset_profiles'])
print('New primitive actions:', report['primitive_interactions'])
check = report['endpoint_reencoding']
print(f"Replay → fresh-render endpoint ranking changes: {check['ranking_changes']}/{check['pairs']}")
print('Replay endpoint accuracy:', sum(r['replay_correct'] for r in check['details']) / check['pairs'])
print('Fresh endpoint accuracy:', sum(r['fresh_correct'] for r in check['details']) / check['pairs'])
if check['ranking_changes']:
    print('Interpretation caveat: fresh rendering/re-encoding changes some fitted endpoint rankings; inspect those pairs before attributing their curves to spatial overfitting.')
print(f"\n{'offset':>7} {'method':>9} {'profiles':>9} {'endpoint OK':>12} {'monotone':>10} {'uphill':>9} {'flat':>6} {'unseen uphill / steps':>25}")
for row in report['offset_summary']:
    for method in ('raw','adapted'):
        m=row[method]
        print(f"{row['offset']:>7g} {method:>9} {m['profiles']:>9} {m['endpoint_accuracy']:>11.1%} "
              f"{m['informative_monotone_fraction']:>9.1%} {m['up_step_fraction']:>8.1%} "
              f"{m['flat_profiles']:>6} {m['unseen_up_steps']:>11}/{m['unseen_steps']:<12}")
print('\nUphill = cost increases despite physical goal distance decreasing.')
print('Unseen comparisons require both endpoints to differ from actual fitting positions and their fresh rendered images.')
print('Conservative replay coverage is also saved per point, including replay frames never sampled by the fitter.')
print(report['note'])
print('\nNearest-fitting-distance bins (step endpoint minimum):')
for row in report['distance_bins']:
    print(row)
print('Fresh execution:', report['fresh_execution'])


## Full-right maps and aggregate generalization

Raw and adapted maps have **independent, unclipped color ranges** so raw structure stays visible. Compare shape and numeric labels, not colors between the two panels. White points mark actual fitting input positions; the goal reference is included.


In [ ]:
for name in ('full_right_heatmaps','offset_generalization','coverage_error'):
    display(Image(filename=str(OUTPUT / 'figures' / f'{name}.png')))


## Individual pairs and nearby positions

The first three pairs come from the predetermined geometry-based selection. Up to three additional examples are the worst original-segment curves, explicitly selected by uphill fraction; these illustrate failures and are not the aggregate sample.

In curve plots, a falling line is the desired direction. Thick lines are original A→B segments; thin lines are shifted segments. Black crosses identify unseen original-segment observations. The pair-relative plot rescales each center curve without changing its direction. Local zooms show measured sample costs, without filling gaps using interpolated costs.


In [ ]:
for name in ('pair_curves','relative_center_curves','local_sample_zooms'):
    display(Image(filename=str(OUTPUT / 'figures' / f'{name}.png')))


## Return the executed notebook

Download this notebook after all cells finish and attach it here. Numeric `report.json` and `probes.json`, plus six PNG/PDF figure pairs, are saved in the printed output folder.

Correct endpoints with interior hills indicate poor interpolation. Good original segments with failures nearby indicate narrow spatial generalization. Good local behavior with lower-band failure points toward a broader extrapolation problem. This fixed-goal test does not measure transfer to other goals.
